In [ ]:
import math


In [15]:
def truncate(num, decimals=2):
    factor = 10.0 ** decimals
    return math.trunc(num * factor) / factor

def load_mz_values(file_path):
    mz_values = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.strip() and not line.lower().startswith('#'):
                try:
                    mz = float(line.strip().split()[0])
                    mz_values.append(mz)
                except ValueError:
                    continue
    return mz_values

def match_mz(sample_list, reference_list):
    matched_sample = set()
    matched_ref = set()

    for sample in sample_list:
        t_sample = truncate(sample, 2)
        r_sample = round(sample, 2)

        found_match = False
        for ref in reference_list:
            t_ref = truncate(ref, 2)
            r_ref = round(ref, 2)

            if math.isclose(t_sample, t_ref):
                matched_sample.add(sample)
                matched_ref.add(ref)
                found_match = True
                break
            elif math.isclose(r_sample, r_ref):
                matched_sample.add(sample)
                matched_ref.add(ref)
                found_match = True
                break
            elif math.isclose(r_sample, t_ref):
                matched_sample.add(sample)
                matched_ref.add(ref)
                found_match = True
                break
            elif math.isclose(t_sample, r_ref):
                matched_sample.add(sample)
                matched_ref.add(ref)
                found_match = True
                break

def match_mz_with_tolerance(sample_list, reference_list, tolerance=0.01):
    matched_sample = set()
    matched_ref = set()

    for sample in sample_list:
        for ref in reference_list:
            if abs(sample - ref) <= tolerance:
                matched_sample.add(sample)
                matched_ref.add(ref)
                break

    unmatched_sample = [m for m in sample_list if m not in matched_sample]
    unmatched_ref = [r for r in reference_list if r not in matched_ref]

    return unmatched_sample, unmatched_ref



In [ ]:
# === File paths ===
reference_file_path = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/first_1000_mz_1000.txt"  # Replace with actual reference file path
sample_file_path = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/first_top_peaks_00001_010da_add_0032da_glob_1000top_sum.txt"      # Replace with actual sample file path
output_file_path = "unmatch/unmatched_results"+


# === Run the comparison ===
sample_mzs = load_mz_values(sample_file_path)
reference_mzs = load_mz_values(reference_file_path)
#unmatched_sample, unmatched_ref = match_mz(sample_mzs, reference_mzs)
unmatched_sample, unmatched_ref = match_mz_with_tolerance(sample_mzs, reference_mzs, tolerance=0.02)


# === Output ===
unmatched_sample.sort()  # Sort in ascending order
unmatched_ref.sort()  # Sort in ascending order


# === Save output to file ===
with open(output_file_path, 'w') as out:
    header = f"ref_mz (n={len(unmatched_ref)})\tsample_mz (n={len(unmatched_sample)})\n"
    out.write(header)
    max_len = max(len(unmatched_sample), len(unmatched_ref))
    for i in range(max_len):
        ref_val = f"{unmatched_ref[i]:.6f}" if i < len(unmatched_ref) else ""
        sample_val = f"{unmatched_sample[i]:.6f}" if i < len(unmatched_sample) else ""
        out.write(f"{ref_val}\t{sample_val}\n")

print(f"✅ Unmatched results saved to '{output_file_path}'")

✅ Unmatched results saved to 'unmatch/unmatched_results.txt'


In [ ]:
'''import os

def load_mz_values(file_path):
    mz_values = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.strip() and not line.lower().startswith('#'):
                try:
                    mz = float(line.strip().split()[0])
                    mz_values.append(mz)
                except ValueError:
                    continue
    return mz_values

def match_mz_with_tolerance(sample_list, reference_list, tolerance=0.01):
    matched_sample = set()
    matched_ref = set()

    for sample in sample_list:
        for ref in reference_list:
            if abs(sample - ref) <= tolerance:
                matched_sample.add(sample)
                matched_ref.add(ref)
                break

    unmatched_sample = sorted([m for m in sample_list if m not in matched_sample])
    unmatched_ref = sorted([r for r in reference_list if r not in matched_ref])

    return unmatched_sample, unmatched_ref'''

In [2]:


# === Setup ===
folder_path = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/txt_files"        # 🔁 Replace with the folder containing .txt files
reference_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/first_1000_mz_1000.txt"              # 🔁 Replace with the name of the reference file
reference_path = os.path.join(folder_path, reference_file)

# Load reference m/z values
reference_mzs = load_mz_values(reference_path)

# Iterate over all .txt files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".txt") and file_name != reference_file:
        sample_path = os.path.join(folder_path, file_name)
        sample_mzs = load_mz_values(sample_path)
        
        unmatched_sample, unmatched_ref = match_mz_with_tolerance(sample_mzs, reference_mzs, tolerance=0.02)

        # Save result with unmatched_<original_filename>.txt
        unmatched_filename = f"unmatched_{file_name}"
        output_path = os.path.join(folder_path, unmatched_filename)
        
        with open(output_path, 'w') as out:
            header = f"ref_mz (n={len(unmatched_ref)})\tsample_mz (n={len(unmatched_sample)})\n"
            out.write(header)

            max_len = max(len(unmatched_sample), len(unmatched_ref))
            for i in range(max_len):
                ref_val = f"{unmatched_ref[i]:.6f}" if i < len(unmatched_ref) else ""
                sample_val = f"{unmatched_sample[i]:.6f}" if i < len(unmatched_sample) else ""
                out.write(f"{ref_val}\t{sample_val}\n")
        
        print(f"✅ Compared '{file_name}' — saved to '{unmatched_filename}'")

✅ Compared 'first_top_peaks.txt' — saved to 'unmatched_first_top_peaks.txt'
✅ Compared 'first_top_peaks_00001_0020da_glob_1000top_sum.txt' — saved to 'unmatched_first_top_peaks_00001_0020da_glob_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_010da_add_0032da_glob_1000top_sum.txt' — saved to 'unmatched_first_top_peaks_00001_010da_add_0032da_glob_1000top_sum.txt'
✅ Compared 'first_top_peaks_1000_0020_max.txt' — saved to 'unmatched_first_top_peaks_1000_0020_max.txt'
✅ Compared 'first_top_peaks_1000_1000_max.txt' — saved to 'unmatched_first_top_peaks_1000_1000_max.txt'
✅ Compared 'first_1000_mz_10000.txt' — saved to 'unmatched_first_1000_mz_10000.txt'
✅ Compared 'first_top_peaks_00001_0024da_glob_1000top_sum.txt' — saved to 'unmatched_first_top_peaks_00001_0024da_glob_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_0050da_glob_1000top_sum.txt' — saved to 'unmatched_first_top_peaks_00001_0050da_glob_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_010da_add_0005da_glob_1000top_sum.

In [1]:
import os

def load_mz_values(file_path):
    mz_values = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.strip() and not line.lower().startswith('#'):
                try:
                    mz = float(line.strip().split()[0])
                    mz_values.append(mz)
                except ValueError:
                    continue
    return mz_values

def match_mz_with_tolerance(sample_list, reference_list, tolerance=0.01):
    matched_sample = set()
    matched_ref = set()

    for sample in sample_list:
        for ref in reference_list:
            if abs(sample - ref) <= tolerance:
                matched_sample.add(sample)
                matched_ref.add(ref)
                break

    unmatched_sample = sorted([m for m in sample_list if m not in matched_sample])
    unmatched_ref = sorted([r for r in reference_list if r not in matched_ref])

    return unmatched_sample, unmatched_ref

In [5]:
# === Setup ===
folder_path = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/txt_files"
reference_file = "first_1000_mz_1000.txt"
reference_path = os.path.join(folder_path, reference_file)

# Create 'unmatched' subfolder if it doesn't exist
unmatched_folder = os.path.join(folder_path, "unmatched")
os.makedirs(unmatched_folder, exist_ok=True)

# Load reference m/z values
reference_mzs = load_mz_values(reference_path)

# Iterate over all .txt files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".txt") and file_name != reference_file:
        sample_path = os.path.join(folder_path, file_name)
        sample_mzs = load_mz_values(sample_path)

        unmatched_sample, unmatched_ref = match_mz_with_tolerance(sample_mzs, reference_mzs, tolerance=0.002)

        # Save result to the 'unmatched' folder
        unmatched_filename = f"unmatched_{file_name}"
        output_path = os.path.join(unmatched_folder, unmatched_filename)

        with open(output_path, 'w') as out:
            header = f"ref_mz (n={len(unmatched_ref)})\tsample_mz (n={len(unmatched_sample)})\n"
            out.write(header)

            max_len = max(len(unmatched_sample), len(unmatched_ref))
            for i in range(max_len):
                ref_val = f"{unmatched_ref[i]:.6f}" if i < len(unmatched_ref) else ""
                sample_val = f"{unmatched_sample[i]:.6f}" if i < len(unmatched_sample) else ""
                out.write(f"{ref_val}\t{sample_val}\n")

        print(f"✅ Compared '{file_name}' — saved to '{output_path}'")

✅ Compared 'first_top_peaks_00001_0.01da_add_0.001da_9w0p_savit_glob_tic_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/txt_files/unmatched/unmatched_first_top_peaks_00001_0.01da_add_0.001da_9w0p_savit_glob_tic_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_0.02da_add_0.001da_15w1p_savit_glob_tic_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/txt_files/unmatched/unmatched_first_top_peaks_00001_0.02da_add_0.001da_15w1p_savit_glob_tic_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_0.05da_add_1e-05da_glob_tic_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/txt_files/unmatched/unmatched_first_top_peaks_00001_0.05da_add_1e-05da_glob_tic_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_0.05da_add_0.01da_11w2p_savit_glob_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/txt_files/unmatched/unmatched_first_top_peaks_00001_0.05da_add_0.01da_11w2p_savit_glob_1000top_sum.t

In [1]:
import os

def load_mz_values(file_path):
    mz_values = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.strip() and not line.lower().startswith('#'):
                try:
                    mz = float(line.strip().split()[0])
                    mz_values.append(mz)
                except ValueError:
                    continue
    return mz_values

def match_mz_with_tolerance(sample_list, reference_list, tolerance=0.01):
    matched_sample = set()
    matched_ref = set()
    used_refs = set()

    for sample in sample_list:
        for ref in reference_list:
            if ref in used_refs:
                continue
            if abs(sample - ref) <= tolerance:
                matched_sample.add(sample)
                matched_ref.add(ref)
                used_refs.add(ref)
                break

    unmatched_sample = sorted(set(sample_list) - matched_sample)
    unmatched_ref = sorted(set(reference_list) - matched_ref)

    return unmatched_sample, unmatched_ref

In [3]:
# === Setup ===
folder_path = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/txt_files"
reference_file = "first_1000_mz_1000.txt"
reference_path = os.path.join(folder_path, reference_file)

# Create 'unmatched' subfolder if it doesn't exist
unmatched_folder = os.path.join(folder_path, "unmatched")
os.makedirs(unmatched_folder, exist_ok=True)

# Load reference m/z values
reference_mzs = load_mz_values(reference_path)

# Iterate over all .txt files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".txt") and file_name != reference_file:
        sample_path = os.path.join(folder_path, file_name)
        sample_mzs = load_mz_values(sample_path)

        unmatched_sample, unmatched_ref = match_mz_with_tolerance(sample_mzs, reference_mzs, tolerance=0.02)

        # Save result to the 'unmatched' folder
        unmatched_filename = f"unmatched_{file_name}"
        output_path = os.path.join(unmatched_folder, unmatched_filename)

        with open(output_path, 'w') as out:
            header = f"ref_mz (n={len(unmatched_ref)})\tsample_mz (n={len(unmatched_sample)})\n"
            out.write(header)

            max_len = max(len(unmatched_sample), len(unmatched_ref))
            for i in range(max_len):
                ref_val = f"{unmatched_ref[i]:.6f}" if i < len(unmatched_ref) else ""
                sample_val = f"{unmatched_sample[i]:.6f}" if i < len(unmatched_sample) else ""
                out.write(f"{ref_val}\t{sample_val}\n")

        print(f"✅ Compared '{file_name}' — saved to '{output_path}'")

✅ Compared 'first_top_peaks_00001_0.01da_add_0.001da_9w0p_savit_glob_tic_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/txt_files/unmatched/unmatched_first_top_peaks_00001_0.01da_add_0.001da_9w0p_savit_glob_tic_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_0.02da_add_0.001da_15w1p_savit_glob_tic_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/txt_files/unmatched/unmatched_first_top_peaks_00001_0.02da_add_0.001da_15w1p_savit_glob_tic_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_0.05da_add_1e-05da_glob_tic_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/txt_files/unmatched/unmatched_first_top_peaks_00001_0.05da_add_1e-05da_glob_tic_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_0.05da_add_0.01da_11w2p_savit_glob_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/txt_files/unmatched/unmatched_first_top_peaks_00001_0.05da_add_0.01da_11w2p_savit_glob_1000top_sum.t

In [1]:
import os

def load_mz_values(file_path):
    mz_values = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.strip() and not line.lower().startswith('#'):
                try:
                    mz = float(line.strip().split()[0])
                    mz_values.append(mz)
                except ValueError:
                    continue
    return mz_values

def match_mz_with_tolerance(sample_list, reference_list, tolerance=0.01):
    matched_sample = set()
    matched_ref = set()
    used_refs = set()

    for sample in sample_list:
        for ref in reference_list:
            if ref in used_refs:
                continue
            if abs(sample - ref) <= tolerance:
                matched_sample.add(sample)
                matched_ref.add(ref)
                used_refs.add(ref)
                break

    unmatched_sample = sorted(set(sample_list) - matched_sample)
    unmatched_ref = sorted(set(reference_list) - matched_ref)

    return unmatched_sample, unmatched_ref

In [2]:
# === Setup ===
folder_path = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/PeakDetection/txt_files"
reference_file = "first_1000_mz_1000.txt"
reference_path = os.path.join(folder_path, reference_file)

tolerance_list = [0.01, 0.02, 0.001, 0.002]  # List of tolerances to test

for tolerance in tolerance_list:
    # Create 'unmatched' subfolder if it doesn't exist
    unmatched_folder = os.path.join(folder_path, "unmatched"+f"_{tolerance}")
    os.makedirs(unmatched_folder, exist_ok=True)

    # Load reference m/z values
    reference_mzs = load_mz_values(reference_path)

    # Iterate over all .txt files in the folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".txt") and file_name != reference_file:
            sample_path = os.path.join(folder_path, file_name)
            sample_mzs = load_mz_values(sample_path)

            unmatched_sample, unmatched_ref = match_mz_with_tolerance(sample_mzs, reference_mzs, tolerance=tolerance)

            # Save result to the 'unmatched' folder
            unmatched_filename = f"unmatched_{file_name}"
            output_path = os.path.join(unmatched_folder, unmatched_filename)

            with open(output_path, 'w') as out:
                header = f"ref_mz (n={len(unmatched_ref)})\tsample_mz (n={len(unmatched_sample)})\n"
                out.write(header)

                max_len = max(len(unmatched_sample), len(unmatched_ref))
                for i in range(max_len):
                    ref_val = f"{unmatched_ref[i]:.6f}" if i < len(unmatched_ref) else ""
                    sample_val = f"{unmatched_sample[i]:.6f}" if i < len(unmatched_sample) else ""
                    out.write(f"{ref_val}\t{sample_val}\n")

            print(f"✅ Compared '{file_name}' — saved to '{output_path}'")

✅ Compared 'first_top_peaks_00001_0.01da_add_0.001da_9w0p_savit_glob_tic_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/PeakDetection/txt_files/unmatched_0.01/unmatched_first_top_peaks_00001_0.01da_add_0.001da_9w0p_savit_glob_tic_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_0.02da_add_0.001da_15w1p_savit_glob_tic_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/PeakDetection/txt_files/unmatched_0.01/unmatched_first_top_peaks_00001_0.02da_add_0.001da_15w1p_savit_glob_tic_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_0.05da_add_1e-05da_glob_tic_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/PeakDetection/txt_files/unmatched_0.01/unmatched_first_top_peaks_00001_0.05da_add_1e-05da_glob_tic_1000top_sum.txt'
✅ Compared 'first_top_peaks_00001_0.05da_add_0.01da_11w2p_savit_glob_1000top_sum.txt' — saved to '/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/PeakDetection/txt_files/unmatched_0.01/unma